# 02 — Sentiment Analysis
Compare VADER and BERT sentiment scores across the dataset and validate against star ratings.

In [ ]:
import sys, pathlib
ROOT = str(pathlib.Path().resolve().parent)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

from src.data_processing.data_loader import load_hotel_reviews
from src.data_processing.data_cleaner import clean_reviews
from src.analysis.sentiment_analyzer import SentimentAnalyzer

SAMPLE = 2000  # set to 0 for full dataset
df = clean_reviews(load_hotel_reviews())
if SAMPLE:
    df = df.sample(SAMPLE, random_state=42).reset_index(drop=True)

print(f'Working with {len(df):,} reviews')

## 1. VADER sentiment scores

In [ ]:
vader = SentimentAnalyzer(method='vader')
df['vader_score'] = vader.analyze_batch(df['review_text'].tolist())
df['vader_label'] = df['vader_score'].apply(vader.get_sentiment_label)
print(df['vader_label'].value_counts())

## 2. Sentiment score vs star rating

In [ ]:
grouped = df.groupby(df['rating'].round(0))['vader_score'].mean()
grouped.plot(kind='bar', figsize=(8, 4), color='teal', edgecolor='white')
plt.xlabel('Star Rating (rounded)')
plt.ylabel('Mean VADER Score')
plt.title('VADER Sentiment vs Guest Star Rating')
plt.axhline(0, color='red', linestyle='--')
plt.tight_layout()
plt.show()

## 3. Label distribution pie chart

In [ ]:
counts = df['vader_label'].value_counts()
colours = {'positive': '#4CAF50', 'neutral': '#FFC107', 'negative': '#F44336'}
plt.figure(figsize=(5, 5))
plt.pie(counts, labels=counts.index, colors=[colours[l] for l in counts.index],
        autopct='%1.1f%%', startangle=90)
plt.title('Sentiment Distribution (VADER)')
plt.tight_layout()
plt.show()

## 4. Accuracy vs star-derived ground truth

In [ ]:
from sklearn.metrics import accuracy_score, classification_report

def star_to_label(r):
    if r >= 3.5: return 'positive'
    if r < 2.5:  return 'negative'
    return 'neutral'

df['true_label'] = df['rating'].apply(star_to_label)
acc = accuracy_score(df['true_label'], df['vader_label'])
print(f'VADER accuracy (vs star label): {acc:.3f}')
print()
print(classification_report(df['true_label'], df['vader_label']))

## 5. Sentiment by nationality (top 10 nationalities)

In [ ]:
top_nat = df['Reviewer_Nationality'].value_counts().head(10).index
nat_df  = df[df['Reviewer_Nationality'].isin(top_nat)]
pivot   = nat_df.groupby(['Reviewer_Nationality', 'vader_label']).size().unstack(fill_value=0)
pivot_pct = pivot.div(pivot.sum(axis=1), axis=0) * 100

pivot_pct[['positive', 'neutral', 'negative']].plot(
    kind='barh', stacked=True, figsize=(9, 5),
    color=['#4CAF50', '#FFC107', '#F44336']
)
plt.xlabel('Percentage')
plt.title('Sentiment Distribution by Reviewer Nationality')
plt.tight_layout()
plt.show()